# Silver　Layer

## 目的・方針

- `workspace.bronze` の生データ（全カラムstring）を読み込み、Silver層として以下を行う
    - 型変換: 文字列から適切な型（数値・日付）へキャスト
    - 重複排除: ビジネスキー単位でdedup
    - データ品質チェック: 必須キーのnull・不正値（負数など）を除去し、除去件数をログ出力
- テーブル間の結合（sales × products × inventory など）はSilver層では行わず、テーブル単位でクレンジングした状態で提供する（結合はGold層の責務とする）
- ソース（Bronze）はフルスナップショット想定のため、Bronzeと同様に書き込みは `overwrite` とする（再実行しても結果が変わらない = 冪等）
- メタデータ列
    - Bronzeの `_ingested_at`（取り込み時刻）はリネージュとして保持する
    - `_silver_processed_at`: Silver処理時刻を新規付与
    - Bronzeの `_source_table`（ファイルパス）はSilverでは不要なため drop


In [ ]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import IntegerType, DateType

BRONZE_SCHEMA = "workspace.bronze"
SILVER_SCHEMA = "workspace.silver"
SILVER_TABLES = ["products", "sales", "inventory"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

In [ ]:
def clean_products(df):
    """
    商品マスタのクレンジング
    - unit_price を IntegerType へキャスト
    - product_id が null の行を除去
    - product_id で重複排除
    """
    df_typed = df.withColumn("unit_price", col("unit_price").cast(IntegerType()))

    df_clean = df_typed.filter(col("product_id").isNotNull()).dropDuplicates(
        ["product_id"]
    )

    return df_clean

In [ ]:
def clean_sales(df):
    """
    売上データのクレンジング
    - quantity, sales_amount を IntegerType へキャスト
    - sale_date を DateType（yyyy-MM-dd）へキャスト
    - sale_id/store_id/product_id が null、または quantity <= 0 の行を除去
    - sale_id で重複排除
    """
    df_typed = (
        df.withColumn("quantity", col("quantity").cast(IntegerType()))
        .withColumn("sales_amount", col("sales_amount").cast(IntegerType()))
        .withColumn("sale_date", col("sale_date").cast(DateType()))
    )

    df_clean = (
        df_typed.filter(
            col("sale_id").isNotNull()
            & col("store_id").isNotNull()
            & col("product_id").isNotNull()
            & (col("quantity") > 0)
        )
        .dropDuplicates(["sale_id"])
    )

    return df_clean

In [ ]:
def clean_inventory(df):
    """
    在庫データのクレンジング
    - stock_quantity を IntegerType、updated_at を DateType へキャスト
    - store_id/product_id が null、または stock_quantity < 0 の行を除去
    - store_id, product_id で重複排除
    """
    df_typed = df.withColumn(
        "stock_quantity", col("stock_quantity").cast(IntegerType())
    ).withColumn("updated_at", col("updated_at").cast(DateType()))

    df_clean = (
        df_typed.filter(
            col("store_id").isNotNull()
            & col("product_id").isNotNull()
            & (col("stock_quantity") >= 0)
        )
        .dropDuplicates(["store_id", "product_id"])
    )

    return df_clean

In [ ]:
CLEAN_FUNCTIONS = {
    "products": clean_products,
    "sales": clean_sales,
    "inventory": clean_inventory,
}


def ingest_to_silver(table_name: str) -> None:
    """
    Bronzeテーブルをクレンジングして Silver テーブルへ書き込む

    silverテーブル _20_ の接頭辞を付与
    """
    source_table = f"{BRONZE_SCHEMA}._10_bronze_{table_name}"
    target_table = f"{SILVER_SCHEMA}._20_silver_{table_name}"

    df_bronze = spark.read.table(source_table)
    before_count = df_bronze.count()

    df_silver = CLEAN_FUNCTIONS[table_name](df_bronze).withColumn(
        "_silver_processed_at", current_timestamp()
    )
    if "_source_table" in df_silver.columns:
        df_silver = df_silver.drop("_source_table")

    after_count = df_silver.count()
    print(
        f"{table_name}: Bronze {before_count}件 -> Silver {after_count}件 "
        f"（除去 {before_count - after_count}件）"
    )

    (
        df_silver.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"Silver書き込み完了: {target_table}")


for table_name in SILVER_TABLES:
    ingest_to_silver(table_name)

In [ ]:
# 書き込み結果の確認
for table_name in SILVER_TABLES:
    df = spark.read.table(f"{SILVER_SCHEMA}._20_silver_{table_name}")
    print(f"{table_name}: {df.count()}件")
    df.printSchema()
    display(df)